In [18]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"비운 후 예약: {torch.cuda.memory_reserved()/1024**3:.1f} GB")

비운 후 예약: 16.1 GB


In [14]:
import torch, gc

# 안전하게 삭제 (없어도 오류 안 남)
if 'model' in dir():
    del model
if 'tokenizer' in dir():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"할당: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"예약: {torch.cuda.memory_reserved()/1024**3:.1f} GB")

할당: 0.0 GB
예약: 16.1 GB


In [1]:
import torch
print(f"할당된 VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"예약된 VRAM: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

할당된 VRAM: 0.00 GB
예약된 VRAM: 0.00 GB


# Qwen3.6-27B

In [2]:
!pip install transformers
!pip install git+https://github.com/huggingface/transformers.git -q
!pip install accelerate -q
!pip install torch==2.5.1 torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu124 \
    --force-reinstall -q
!pip install bitsandbytes -q
!pip install -U bitsandbytes>=0.46.1 -q
!pip install time

ERROR: Could not find a version that satisfies the requirement time (from versions: none)
ERROR: No matching distribution found for time


In [3]:
import os, transformers

trans_path = os.path.dirname(transformers.__file__)

def patch_file(filepath, target, replacement_lines):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    new = []
    for line in lines:
        if target in line and 'try:' not in line:
            indent = ' ' * (len(line) - len(line.lstrip()))
            for r in replacement_lines: new.append(indent + r)
        else:
            new.append(line)
    with open(filepath, 'w') as f:
        f.writelines(new)

# 패치 1: fsdp.py - CPUOffloadPolicy fallback
patch_file(f"{trans_path}/distributed/fsdp.py",
    "from torch.distributed.fsdp import CPUOffloadPolicy, MixedPrecisionPolicy, OffloadPolicy",
    ["try:\n",
     "    from torch.distributed.fsdp import CPUOffloadPolicy, MixedPrecisionPolicy, OffloadPolicy\n",
     "except ImportError:\n",
     "    CPUOffloadPolicy = MixedPrecisionPolicy = OffloadPolicy = None\n"])
print("✅ 패치 1: fsdp.py")

# 패치 2: continuous_batching
patch_file(f"{trans_path}/generation/continuous_batching/distributed.py",
    "from torch.distributed.tensor.device_mesh import DeviceMesh",
    ["try:\n",
     "    from torch.distributed.tensor.device_mesh import DeviceMesh\n",
     "except ImportError:\n",
     "    from torch.distributed.device_mesh import DeviceMesh\n"])
print("✅ 패치 2: continuous_batching")

# 패치 3: sharding_utils stub
with open(f"{trans_path}/distributed/sharding_utils.py", 'w') as f:
    f.write("""# Stub for PyTorch < 2.6
class DtensorShardOperation:
    pass
def _dtensor_from_local_like(*a, **k): raise NotImplementedError
def _find_strided_shard_placement_from_fused_params(*a, **k): raise NotImplementedError
def _replicate_dtensor(*a, **k): raise NotImplementedError
def fuse_optimizer_state(*a, **k): raise NotImplementedError
def get_fusion_metadata(*a, **k): raise NotImplementedError
def unfuse_optimizer_state(*a, **k): raise NotImplementedError
""")
print("✅ 패치 3: sharding_utils")

print("\n🎉 완료 - 커널 재시작 후 모델 로드!")

✅ 패치 1: fsdp.py
✅ 패치 2: continuous_batching
✅ 패치 3: sharding_utils

🎉 완료 - 커널 재시작 후 모델 로드!


In [4]:
import os, transformers

trans_path = os.path.dirname(transformers.__file__)

# ── tensor_parallel.py 원래 경로로 복구 ──
p1 = f"{trans_path}/distributed/tensor_parallel.py"
with open(p1, 'r') as f:
    content = f.read()

# _tensor → tensor 로 되돌리기
content = (content
    .replace("from torch.distributed._tensor import DTensor, Partial, Replicate, Shard, distribute_tensor",
             "from torch.distributed.tensor import DTensor, Partial, Replicate, Shard, distribute_tensor")
    .replace("from torch.distributed._tensor.parallel import (",
             "from torch.distributed.tensor.parallel import (")
    .replace("from torch.distributed._tensor.parallel.style import ParallelStyle",
             "from torch.distributed.tensor.parallel.style import ParallelStyle")
)

with open(p1, 'w') as f:
    f.write(content)
print("✅ tensor_parallel.py 복구 완료")

# ── ParallelStyle fallback 추가 (없으면) ──
with open(p1, 'r') as f:
    lines = f.readlines()
if not any('class ParallelStyle' in l for l in lines):
    new = []
    for line in lines:
        if 'class TensorParallelStyle(ParallelStyle):' in line:
            new.append("\ntry:\n    from torch.distributed.tensor.parallel.style import ParallelStyle\nexcept ImportError:\n    class ParallelStyle:\n        pass\n\n")
        new.append(line)
    with open(p1, 'w') as f:
        f.writelines(new)
    print("✅ ParallelStyle fallback 추가")

print("🎉 완료 - 커널 재시작 후 모델 로드!")

✅ tensor_parallel.py 복구 완료
🎉 완료 - 커널 재시작 후 모델 로드!


In [5]:
import os, transformers

trans_path = os.path.dirname(transformers.__file__)

def patch_file(filepath, target, replacement_lines):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    new = []
    for line in lines:
        if target in line and 'try:' not in line:
            indent = ' ' * (len(line) - len(line.lstrip()))
            for r in replacement_lines: new.append(indent + r)
        else:
            new.append(line)
    with open(filepath, 'w') as f:
        f.writelines(new)

patch_file(f"{trans_path}/distributed/utils.py",
    "from torch.distributed.checkpoint.hf_storage import HuggingFaceStorageWriter",
    ["try:\n","    from torch.distributed.checkpoint.hf_storage import HuggingFaceStorageWriter\n","except ImportError:\n","    HuggingFaceStorageWriter = None\n"])

patch_file(f"{trans_path}/generation/continuous_batching/distributed.py",
    "from torch.distributed.tensor.device_mesh import DeviceMesh",
    ["try:\n","    from torch.distributed.tensor.device_mesh import DeviceMesh\n","except ImportError:\n","    from torch.distributed.device_mesh import DeviceMesh\n"])

with open(f"{trans_path}/distributed/sharding_utils.py", 'w') as f:
    f.write("""# Stub
class DtensorShardOperation:
    pass
def _dtensor_from_local_like(*a, **k): raise NotImplementedError
def _find_strided_shard_placement_from_fused_params(*a, **k): raise NotImplementedError
def _replicate_dtensor(*a, **k): raise NotImplementedError
def fuse_optimizer_state(*a, **k): raise NotImplementedError
def get_fusion_metadata(*a, **k): raise NotImplementedError
def unfuse_optimizer_state(*a, **k): raise NotImplementedError
""")
print("✅ 패치 완료 - 이제 모델 로드 셀 실행하세요")

✅ 패치 완료 - 이제 모델 로드 셀 실행하세요


In [6]:
import torch
print(f"할당: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print(f"예약: {torch.cuda.memory_reserved()/1024**3:.1f} GB")
print(f"전체: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

할당: 0.0 GB
예약: 0.0 GB
전체: 44.4 GB


In [7]:
import json
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import time

meeting_files = [
    "개발자회의.txt",
    "마케팅회의.txt",
    "인사처회의.txt"
]

# 4-bit 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # 품질 가장 좋은 4bit
    bnb_4bit_compute_dtype="bfloat16", # 연산은 bf16으로
    bnb_4bit_use_double_quant=True,    # 추가 압축
)

model_name = "Qwen/Qwen3.6-27B"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",  # 전체를 GPU 0번에 강제 할당
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print("✅ 4-bit 양자화 모델 로드 완료")

def extract_tasks(meeting_text):
    prompt = f"""[역할]
    너는 회의를 화자별로 분석한 원본 데이터를 분석하여 회의록을 작성하고, 해야 할 일 즉 태스크를 추출하는 Jira AI야 
    한눈에 읽어도 누구나 바로 알 수 있을 정도로 완벽한 정리를 해내지.  해당 내용들은 jira 이슈에도 등록될 예정이야 그러니
    매우 깔끔하고 정확하게 추출해야해 중국어 일본어 절대 쓰지마.
    
    [미션]
    제공된 대화 텍스트를 분석하여 반드시 아래 JSON 형식으로만 응답해줘.
    텍스트에 명시되지 않은 정보는 억지로 지어내지 말고 "미정" 또는 "없음"으로 처리해
    모든 내용은 반드시 100% 한국어(Korean)로만 작성해라. 중국어나 일본어는 절대 사용 금지.

    [담당자 추출 규칙 - 매우 중요]
    회의록은 "이름: 발언내용" 형식으로 구성되어 있어.
    담당자를 결정할 때 아래 두 가지 패턴을 반드시 구분해:

    패턴 1 - 본인이 직접 선언하는 경우 → 발언한 화자가 담당자
      예) "김규호: 저는 발표 자료 만들겠습니다" → assignee: 김규호
      예) "류지우: 저는 STT 테스트 해볼게요" → assignee: 류지우
      예) "박수영: 제가 디자인 맡을게요" → assignee: 박수영

    패턴 2 - 리더가 다른 사람에게 지시하는 경우 → 지시받은 사람이 담당자
      예) "김지원: 규호님은 체크리스트 부탁드립니다" → assignee: 김규호
      예) "김지원: 민준님이 공고 수정해주세요" → assignee: 김민준
    
    [출력 형식]
    {{
      "cotent": "회의록내용을 정리해줘 누가 봐도 한눈에 이해할 정도로 써야해 그러니 너무 간략하게 쓰면 이해하기 못하겠지? 그리고 줄바꿈은 \\n으로 표현하고 JSON 문자열 안에서 실제 줄바꿈 금지. 형식: 큰 주제\\n1. 세부 내용\\n  - 더 구체적인 내용.", 
      "todo_list": [
        {{
          "title": "해당 담당자가 해야 할 업무 내용(태스크 명)",
          "content": "해당 담당자가 해야 할 구체적인 업무 내용",
          "owner": "담당자 이름",
          "due_date": "마감 일정(정확한 마감일을 기입해야해 오늘 날짜 기준으로 마감 일정 설정해줘)",
          "priority": "우선 순위(High, Medium, Low, Lowest)에서 고르도록 해"
        }}
      ]
    }}
    
    [회의록 텍스트]
    {meeting_text}
    """
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Always respond in valid JSON format only, with no extra text."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False  # ← thinking 모드 비활성화
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2048,
        do_sample=False,        # greedy decoding → JSON 안정성 향상
        repetition_penalty=1.1
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # thinking 블록 제거
    import re
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    print(f"  📝 모델 출력: {response[:200]}")  # 디버깅용

    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
        else:
            print(f"⚠️ JSON 없음. 전체 출력:\n{response}")
            return None
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 파싱 실패: {e}\n출력:\n{response}")
        return None


# 전체 파일 처리
all_results = {}
total_start = time.time()  # 전체 시작

for file_name in meeting_files:
    print(f"\n📄 처리 중: {file_name}")

    file_start = time.time()  # 파일별 시작

    with open(file_name, "r", encoding="utf-8") as f:
        meeting_text = f.read()

    result = extract_tasks(meeting_text)

    file_elapsed = time.time() - file_start  # 파일별 종료

    if result:
        all_results[file_name] = result
        print(f"  ✅ 완료 - TODO {len(result['todo_list'])}개 추출 | ⏱ {file_elapsed:.1f}초")
    else:
        print(f"  ❌ 실패: {file_name} | ⏱ {file_elapsed:.1f}초")

total_elapsed = time.time() - total_start  # 전체 종료

with open("output_all.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ output_all.json 저장 완료")
print(f"⏱ 전체 처리 시간: {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

✅ 4-bit 양자화 모델 로드 완료

📄 처리 중: 개발자회의.txt
  📝 모델 출력: {
  "cotent": "공공기관 AI 인프라 구축 사업 RFP 대응 전략 및 개발 현황 공유 회의\n1. STT(Speech-to-Text) 모듈 최적화\n   - WhisperX 기반 테스트 진행 중이며, FastWhisper와 비교 실험 병행\n   - RTX 4090 환경에서 배치 사이즈 조절하며 처리 속도 측정\n   - 오버랩(겹쳐짐) 구간에서
  ✅ 완료 - TODO 9개 추출 | ⏱ 172.6초

📄 처리 중: 마케팅회의.txt
  📝 모델 출력: {
  "cotent": "서비스 마케팅 방향 및 초기 홍보 전략 논의\n1. 핵심 메시지 정의\n  - '회의 후 정리 시간 절감'을 주요 가치 제안으로 설정.\n  - 직장인의 회의 스트레스(정리, 일정 등록 등) 공감을 통한 마케팅 접근.\n  - 메인 문구: '녹음만 하면 회의록 완성', '자동으로 업무 담당자와 일정 정리'.\n2. 타겟 고객 및 
  ✅ 완료 - TODO 4개 추출 | ⏱ 84.1초

📄 처리 중: 인사처회의.txt
  📝 모델 출력: {
  "cotent": "3분기 채용 현황 및 신규 입사자 온보딩, 조직 개편 관련 논의\n1. 채용 현황 및 전략\n  - 총 130명 지원, 개발팀(특히 백엔드/AI/데이터) 경쟁률 높음.\n  - 포트폴리오와 실제 역량 괴리 존재, 협업 능력 및 문제 해결 과정 평가 강화.\n  - 1차 면접은 온라인, 최종 면접은 오프라인으로 진행하여 지방 지원자
  ✅ 완료 - TODO 8개 추출 | ⏱ 137.2초

✅ output_all.json 저장 완료
⏱ 전체 처리 시간: 393.9초 (6.6분)


In [11]:
# 파일 내용 직접 출력
with open("output_all.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for file_name, result in data.items():
    print(f"\n📄 {file_name}")
    print(f"  요약: {result['summary'][:100]}")
    print(f"  TODO: {len(result['todo_list'])}개")


📄 개발자회의.txt
  요약: 공공기관 AI 인프라 구축 사업 RFP 대응을 위해 STT, RAG, 문서 처리 등 핵심 모듈의 기술 검증 현황과 통합 방안을 논의함. 특히 STT의 오버랩 구간 인식 개선, 하이
  TODO: 9개

📄 인사처회의.txt
  요약: 3분기 채용 현황(130명 지원, 백엔드/AI 분야 경쟁 치열) 및 면접 프로세스 개선(1차 온라인/최종 오프라인)을 논의했습니다. 신규 입사자 온보딩 자료의 친근함 강화 및 FA
  TODO: 8개


------

# gemma 4

In [ ]:
# # 젬마 4
# import json
# import re
# import torch
# from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
# import time

# MODEL_ID = "google/gemma-4-E4B-it"
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# # Load model
# processor = AutoProcessor.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     quantization_config=quantization_config,
#     dtype="auto",
#     device_map="cuda:0"
# )

# meeting_files = [
#     "개발자회의.txt",
#     "마케팅회의.txt",
#     "인사처회의.txt"
# ]

# def extract_tasks(meeting_text):
#     prompt = f"""[역할]
#     너는 회의를 화자별로 분석한 원본 데이터를 분석하여 회의록을 작성하고, 해야 할 일 즉 태스크를 추출하는 Jira AI야 
#     한눈에 읽어도 누구나 바로 알 수 있을 정도로 완벽한 정리를 해내지.  해당 내용들은 jira 이슈에도 등록될 예정이야 그러니
#     매우 깔끔하고 정확하게 추출해야해 중국어 일본어 절대 쓰지마.
    
#     [미션]
#     제공된 대화 텍스트를 분석하여 반드시 아래 JSON 형식으로만 응답해줘.
#     텍스트에 명시되지 않은 정보는 억지로 지어내지 말고 "미정" 또는 "없음"으로 처리해
#     모든 내용은 반드시 100% 한국어(Korean)로만 작성해라. 중국어나 일본어는 절대 사용 금지.

#     [담당자 추출 규칙 - 매우 중요]
#     회의록은 "이름: 발언내용" 형식으로 구성되어 있어.
#     담당자를 결정할 때 아래 두 가지 패턴을 반드시 구분해:

#     패턴 1 - 본인이 직접 선언하는 경우 → 발언한 화자가 담당자
#       예) "김규호: 저는 발표 자료 만들겠습니다" → assignee: 김규호
#       예) "류지우: 저는 STT 테스트 해볼게요" → assignee: 류지우
#       예) "박수영: 제가 디자인 맡을게요" → assignee: 박수영

#     패턴 2 - 리더가 다른 사람에게 지시하는 경우 → 지시받은 사람이 담당자
#       예) "김지원: 규호님은 체크리스트 부탁드립니다" → assignee: 김규호
#       예) "김지원: 민준님이 공고 수정해주세요" → assignee: 김민준
    
#     [출력 형식]
#     {{
#       "cotent": "회의록내용을 정리해줘 누가 봐도 한눈에 이해할 정도로 써야해 그러니 너무 간략하게 쓰면 이해하기 못하겠지? 그리고 반드시 아래처럼 작성해. 
#     큰 주제
#     1 세부 내용
#       - 더 구체적인 내용
#     큰 주제
#     - 세부 내용",
#       "todo_list": [
#         {{
#           "title": "해당 담당자가 해야 할 업무 내용(태스크 명)",
#           "content": "해당 담당자가 해야 할 구체적인 업무 내용",
#           "owner": "담당자 이름",
#           "due_date": "마감 일정(정확한 마감일을 기입해야해 오늘 날짜 기준으로 마감 일정 설정해줘)",
#           "priority": "우선 순위(High, Medium, Low, Lowest)에서 고르도록 해"
#         }}
#       ]
#     }}
    
#     [회의록 텍스트]
#     {meeting_text}
#     """

#     messages = [
#         {"role": "system", "content": "너는 한국어 비즈니스 회의록을 분석하여 시스템 데이터로 변환하는 전문가야."},
#         {"role": "user", "content": prompt},
#     ]

#     # 2. Gemma 4 템플릿 적용 및 텐서 변환 (회원님 코드 완벽함!)
#     text = processor.apply_chat_template(
#         messages, 
#         tokenize=False, 
#         add_generation_prompt=True, 
#         enable_thinking=False
#     )
#     inputs = processor(text=text, return_tensors="pt").to(model.device)
#     input_len = inputs["input_ids"].shape[-1]
    
#     # 3. 텍스트 생성
#     print("Gemma 4가 회의록을 분석하고 있습니다...")
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens=2048,
#         temperature=0.3,  # JSON 추출 시 온도를 낮추면 더 정확해집니다
#         do_sample=False 
#     )

#     # 🚨 수정 1: 특수 토큰 제거 (True로 변경)
#     response = processor.decode(outputs[0][input_len:], skip_special_tokens=True)
#     # JSON 파싱
#     try:
#         json_match = re.search(r'\{.*\}', response, re.DOTALL)
#         if json_match:
#             return json.loads(json_match.group())
#         else:
#             raise ValueError("JSON 블록을 찾을 수 없음")
#     except json.JSONDecodeError:
#         print(f"⚠️ JSON 파싱 실패, 원본 출력:\n{response}")
#         return None

# # 전체 파일 처리
# all_results = {}
# total_start = time.time()  # 전체 시작

# for file_name in meeting_files:
#     print(f"\n📄 처리 중: {file_name}")

#     file_start = time.time()  # 파일별 시작

#     with open(file_name, "r", encoding="utf-8") as f:
#         meeting_text = f.read()

#     result = extract_tasks(meeting_text)

#     file_elapsed = time.time() - file_start  # 파일별 종료

#     if result:
#         all_results[file_name] = result
#         print(f"  ✅ 완료 - TODO {len(result['todo_list'])}개 추출 | ⏱ {file_elapsed:.1f}초")
#     else:
#         print(f"  ❌ 실패: {file_name} | ⏱ {file_elapsed:.1f}초")

# total_elapsed = time.time() - total_start  # 전체 종료

# with open("output_all.json", "w", encoding="utf-8") as f:
#     json.dump(all_results, f, ensure_ascii=False, indent=2)

# print(f"\n✅ output_all.json 저장 완료")
# print(f"⏱ 전체 처리 시간: {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

# gpt4.1-mini

In [ ]:
# !pip install dotenv
# !pip install openai

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]
  Using cached openai-2.38.0-py3-none-any.whl.metadata (31 kB)
  Using cached jiter-0.15.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.2 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached openai-2.38.0-py3-none-any.whl (1.3 MB)
Using cached jiter-0.15.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (346 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 

In [ ]:
# from pathlib import Path
# from dotenv import load_dotenv
# from openai import OpenAI
# import json, os

# load_dotenv()
# client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# def extract_tasks_with_llm(transcript):
#     prompt = f"""[역할]
#     너는 회의를 화자별로 분석한 원본 데이터를 분석하여 회의록을 작성하고, 해야 할 일 즉 태스크를 추출하는 Jira AI야 
#     한눈에 읽어도 누구나 바로 알 수 있을 정도로 완벽한 정리를 해내지.  해당 내용들은 jira 이슈에도 등록될 예정이야 그러니
#     매우 깔끔하고 정확하게 추출해야해 중국어 일본어 절대 쓰지마.
    
#     [미션]
#     제공된 대화 텍스트를 분석하여 반드시 아래 JSON 형식으로만 응답해줘.
#     텍스트에 명시되지 않은 정보는 억지로 지어내지 말고 "미정" 또는 "없음"으로 처리해
#     모든 내용은 반드시 100% 한국어(Korean)로만 작성해라. 중국어나 일본어는 절대 사용 금지.
#     [담당자 추출 규칙 - 매우 중요]
#     회의록은 "이름: 발언내용" 형식으로 구성되어 있어.
#     담당자를 결정할 때 아래 두 가지 패턴을 반드시 구분해:
#     패턴 1 - 본인이 직접 선언하는 경우 → 발언한 화자가 담당자
#       예) "김규호: 저는 발표 자료 만들겠습니다" → assignee: 김규호
#       예) "류지우: 저는 STT 테스트 해볼게요" → assignee: 류지우
#       예) "박수영: 제가 디자인 맡을게요" → assignee: 박수영
#     패턴 2 - 리더가 다른 사람에게 지시하는 경우 → 지시받은 사람이 담당자
#       예) "김지원: 규호님은 체크리스트 부탁드립니다" → assignee: 김규호
#       예) "김지원: 민준님이 공고 수정해주세요" → assignee: 김민준
    
#     [출력 형식]
#     {{
#       "cotent": "회의록내용을 정리해줘 누가 봐도 한눈에 이해할 정도로 써야해 그러니 너무 간략하게 쓰면 이해하기 못하겠지? 그리고 반드시 아래처럼 작성해. 큰 주제 1 세부 내용 - 더 구체적인 내용 큰 주제 - 세부 내용",
#       "todo_list": [
#         {{
#           "title": "해당 담당자가 해야 할 업무 내용(태스크 명)",
#           "content": "해당 담당자가 해야 할 구체적인 업무 내용",
#           "owner": "담당자 이름",
#           "due_date": "마감 일정(정확한 마감일을 기입해야해 오늘 날짜 기준으로 마감 일정 설정해줘)",
#           "priority": "우선 순위(High, Medium, Low, Lowest)에서 고르도록 해"
#         }}
#       ]
#     }}
    
#     [회의록 텍스트]
#     {transcript}
#     """

#     response = client.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {
#                 "role": "system",
#                 "content": "You are a helpful assistant. Always respond in valid JSON format only, with no extra text. 마크다운 코드블록 사용 금지."
#             },
#             {
#                 "role": "user",
#                 "content": prompt
#             }
#         ],
#         temperature=0
#     )
#     content = response.choices[0].message.content.strip()
#     content = content.replace("```json", "").replace("```", "").strip()
#     return json.loads(content)


# file_pairs = [
#     ("개발자회의.txt",  "meeting_result(개발).json"),
#     ("마케팅회의.txt",  "meeting_result(마케팅).json"),
#     ("인사처회의.txt",  "meeting_result(인사처).json"),
# ]

# total_start = time.time()  # ← 전체 시작

# for input_file, output_file in file_pairs:
#     print(f"\n📄 처리 중: {input_file}")
#     file_start = time.time()  # ← 파일별 시작
#     try:
#         transcript = Path(input_file).read_text(encoding="utf-8")
#         result = extract_tasks_with_llm(transcript)
#         output_json = {input_file: result}
#         json_text = json.dumps(output_json, ensure_ascii=False, indent=2)
#         Path(output_file).write_text(json_text, encoding="utf-8")
#         elapsed = time.time() - file_start  # ← 파일별 종료
#         print(f"  ✅ 완료 - TODO {len(result.get('todo_list', []))}개 → {output_file} | ⏱ {elapsed:.1f}초")
#     except FileNotFoundError:
#         print(f"  ❌ 파일 없음: {input_file}")
#     except Exception as e:
#         elapsed = time.time() - file_start
#         print(f"  ❌ 오류: {e} | ⏱ {elapsed:.1f}초")

# total_elapsed = time.time() - total_start  # ← 전체 종료
# print(f"\n⏱ 전체 처리 시간: {total_elapsed:.1f}초 ({total_elapsed/60:.1f}분)")


📄 처리 중: 개발자회의.txt
  ✅ 완료 - TODO 10개 → meeting_result(개발).json | ⏱ 17.3초

📄 처리 중: 마케팅회의.txt
  ✅ 완료 - TODO 5개 → meeting_result(마케팅).json | ⏱ 6.9초

📄 처리 중: 인사처회의.txt
  ✅ 완료 - TODO 10개 → meeting_result(인사처).json | ⏱ 15.5초

⏱ 전체 처리 시간: 39.7초 (0.7분)
